<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
documents = [
    "Artificial intelligence enables computers to perform tasks that normally require human intelligence.",
    "Machine learning allows computers to learn patterns from data without being explicitly programmed.",
    "Deep learning is a branch of machine learning that uses neural networks with multiple layers.",
    "Natural language processing enables computers to understand and process human language.",
    "Computer vision allows machines to understand and analyze images and videos.",
    "Neural networks are computing models inspired by the structure of the human brain.",
    "Supervised learning trains a model using labeled training data.",
    "Unsupervised learning discovers hidden patterns in data without labeled examples.",
    "Reinforcement learning trains an agent through rewards and penalties.",
    "Classification algorithms assign data into predefined categories.",
    "Regression algorithms predict continuous numerical values from input data.",
    "Clustering groups similar data points together based on their characteristics.",
    "Training data is used to teach a machine learning model how to make predictions.",
    "Testing data is used to evaluate how well a trained machine learning model performs.",
    "Overfitting happens when a machine learning model learns the training data too closely.",
    "Feature engineering involves creating useful input features from raw data.",
    "Artificial neural networks can be used for image recognition and speech recognition.",
    "Generative AI can create new text, images, audio, and other types of content.",
    "Large language models are AI models trained on large amounts of text to understand and generate language.",
    "AI applications are widely used in healthcare, finance, education, transportation, and cybersecurity."
]

In [2]:
print("Number of documents:", len(documents))

Number of documents: 20


In [5]:
class Pipeline:

    def __init__(self, preprocessor, vectorizer):
        self.preprocessor = preprocessor
        self.vectorizer = vectorizer

    def run(self, query, corpus):

        # Validate query
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query cannot be empty.")

        # Preprocess query
        processed_query = self.preprocessor.transform(query)

        # Preprocess all documents
        processed_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        # Fit vectorizer using processed corpus
        self.vectorizer.fit(processed_corpus)

        # Calculate similarity scores
        scores = self.vectorizer.transform(processed_query)

        # Create ranked results
        ranked_results = []

        for index, score in enumerate(scores):
            ranked_results.append({
                "document_id": index + 1,
                "document": corpus[index],
                "similarity_score": float(score)
            })

        # Sort from highest similarity to lowest
        ranked_results.sort(
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        return ranked_results

In [10]:
class PreprocessingModule:

    def __init__(self):
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Input text cannot be empty.")

        text = text.lower()

        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'\s+', ' ', text).strip()

        if not text:
            raise ValueError("Input contains no meaningful text.")

        tokens = text.split()

        tokens = [
            word for word in tokens
            if word not in self.stop_words
        ]

        tokens = [
            self.lemmatizer.lemmatize(word)
            for word in tokens
        ]

        if not tokens:
            raise ValueError("Input contains no meaningful words.")

        return " ".join(tokens)

In [11]:
class VectorizerModule:

    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None
        self.corpus = None

    def fit(self, corpus):
        self.corpus = corpus
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        if self.corpus_vectors is None:
            raise ValueError(
                "Vectorizer must be fitted before transforming a query."
            )

        query_vector = self.vectorizer.transform([query])

        similarity_scores = cosine_similarity(
            query_vector,
            self.corpus_vectors
        )[0]

        return similarity_scores

In [12]:
class Pipeline:

    def __init__(self, preprocessor, vectorizer):
        self.preprocessor = preprocessor
        self.vectorizer = vectorizer

    def run(self, query, corpus):

        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query cannot be empty.")

        processed_query = self.preprocessor.transform(query)

        processed_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        self.vectorizer.fit(processed_corpus)

        scores = self.vectorizer.transform(processed_query)

        ranked_results = []

        for index, score in enumerate(scores):
            ranked_results.append({
                "document_id": index + 1,
                "document": corpus[index],
                "similarity_score": float(score)
            })

        ranked_results.sort(
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        return ranked_results

In [14]:
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [17]:
preprocessor = PreprocessingModule()
vectorizer = VectorizerModule()

pipeline = Pipeline(
    preprocessor,
    vectorizer
)

print("Pipeline created successfully!")

Pipeline created successfully!


In [18]:
processed_documents = [
    preprocessor.transform(document)
    for document in documents
]

print("Number of processed documents:", len(processed_documents))
print("First processed document:")
print(processed_documents[0])

Number of processed documents: 20
First processed document:
artificial intelligence enables computer perform task normally require human intelligence


In [19]:
vectorizer.fit(processed_documents)

tfidf_matrix = vectorizer.corpus_vectors

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (20, 109)


In [20]:
def retrieve(query, corpus_matrix, top_k=3):

    # Preprocess the query
    processed_query = preprocessor.transform(query)

    # Convert query into TF-IDF vector
    query_vector = vectorizer.vectorizer.transform([processed_query])

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    # Get document indices sorted by similarity
    ranked_indices = similarity_scores.argsort()[::-1]

    # Select top K documents
    top_indices = ranked_indices[:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document_id": index + 1,
            "document": documents[index],
            "similarity_score": float(similarity_scores[index])
        })

    return results

In [21]:
results = retrieve(
    "What is machine learning?",
    tfidf_matrix,
    top_k=3
)

for result in results:
    print("Document ID:", result["document_id"])
    print("Document:", result["document"])
    print("Similarity Score:", round(result["similarity_score"], 3))
    print("-" * 60)

Document ID: 3
Document: Deep learning is a branch of machine learning that uses neural networks with multiple layers.
Similarity Score: 0.427
------------------------------------------------------------
Document ID: 13
Document: Training data is used to teach a machine learning model how to make predictions.
Similarity Score: 0.356
------------------------------------------------------------
Document ID: 15
Document: Overfitting happens when a machine learning model learns the training data too closely.
Similarity Score: 0.342
------------------------------------------------------------


In [22]:
def retrieve(query, corpus_matrix, top_k=3):

    # Preprocess the query
    processed_query = preprocessor.transform(query)

    # Convert query into TF-IDF vector
    query_vector = vectorizer.vectorizer.transform([processed_query])

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    # Find highest similarity score
    highest_score = similarity_scores.max()

    # Relevance threshold
    if highest_score < 0.1:
        return "No relevant document found"

    # Rank documents from highest to lowest similarity
    ranked_indices = similarity_scores.argsort()[::-1]

    # Select top K
    top_indices = ranked_indices[:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document_id": index + 1,
            "document": documents[index],
            "similarity_score": float(similarity_scores[index])
        })

    return results

In [23]:
results = retrieve(
    "What is machine learning?",
    tfidf_matrix,
    top_k=3
)

print(results)

[{'document_id': np.int64(3), 'document': 'Deep learning is a branch of machine learning that uses neural networks with multiple layers.', 'similarity_score': 0.42731282605074494}, {'document_id': np.int64(13), 'document': 'Training data is used to teach a machine learning model how to make predictions.', 'similarity_score': 0.3561550674531298}, {'document_id': np.int64(15), 'document': 'Overfitting happens when a machine learning model learns the training data too closely.', 'similarity_score': 0.3417560050668056}]


In [24]:
results = retrieve(
    "How do airplanes fly?",
    tfidf_matrix,
    top_k=3
)

print(results)

No relevant document found


In [26]:
test_queries = [
    # Normal queries
    "What is machine learning?",
    "How does deep learning work?",
    "What is natural language processing?",
    "What is computer vision?",
    "What is reinforcement learning?",
    "How are neural networks used?",

    # Ambiguous queries
    "Model",
    "Training",

    # Out-of-domain queries
    "How do airplanes fly?",
    "What is the capital of France?"
]

In [27]:
for i, query in enumerate(test_queries, start=1):

    print("\n" + "=" * 70)
    print(f"QUERY {i}: {query}")
    print("=" * 70)

    results = retrieve(
        query,
        tfidf_matrix,
        top_k=3
    )

    if isinstance(results, str):
        print(results)

    else:
        for rank, result in enumerate(results, start=1):
            print(
                f"{rank}. Document {result['document_id']} "
                f"| Score: {result['similarity_score']:.3f}"
            )
            print(f"   {result['document']}")


QUERY 1: What is machine learning?
1. Document 3 | Score: 0.427
   Deep learning is a branch of machine learning that uses neural networks with multiple layers.
2. Document 13 | Score: 0.356
   Training data is used to teach a machine learning model how to make predictions.
3. Document 15 | Score: 0.342
   Overfitting happens when a machine learning model learns the training data too closely.

QUERY 2: How does deep learning work?
1. Document 3 | Score: 0.502
   Deep learning is a branch of machine learning that uses neural networks with multiple layers.
2. Document 9 | Score: 0.118
   Reinforcement learning trains an agent through rewards and penalties.
3. Document 7 | Score: 0.118
   Supervised learning trains a model using labeled training data.

QUERY 3: What is natural language processing?
1. Document 4 | Score: 0.724
   Natural language processing enables computers to understand and process human language.
2. Document 19 | Score: 0.253
   Large language models are AI models trai

In [28]:
retrieve(query, tfidf_matrix, top_k=3)

'No relevant document found'